# Nextflow work directory debug

Inspect failed tasks under a GoodWorkflows run `work/` folder.

**Kernel:** use [GoodWorkflows (notebooks venv)](../README.md#local-python-environment-and-kernel) — see `notebooks/README.md` for `venv` setup.

Copy this file to `notebooks/examples/` before editing parameters.

In [ ]:
from pathlib import Path
import sys

# --- Parameters (edit per session) ---
RUN_DIR_OVERRIDE = None  # e.g. Path("/abs/path/to/runs/my_run")

def _bootstrap_lib() -> None:
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        lib = base / "notebooks" / "lib"
        if (lib / "workdir_utils.py").is_file():
            sys.path.insert(0, str(lib))
            return
    raise ImportError("Could not find notebooks/lib/workdir_utils.py")

_bootstrap_lib()
import workdir_utils as wu

REPO_ROOT = wu.find_repo_root()
RUN_DIR = RUN_DIR_OVERRIDE or (REPO_ROOT / "template" / "gw" / "runs" / "latest")

WORK_DIR = RUN_DIR / "work"
LOG_FILE = RUN_DIR / "logs" / "nextflow.log"
OUTPUTS_DIR = RUN_DIR / "outputs"

paths = wu.validate_run_layout(RUN_DIR)
print("Run dir:", paths["run_dir"])
print("Work dir:", paths["work_dir"])
print("Outputs:", paths["outputs_dir"])
print("Log exists:", paths["log_file"].is_file(), "->", paths["log_file"])

In [ ]:
tasks = wu.list_task_dirs(paths["work_dir"])
print(f"Found {len(tasks)} task work directories (newest first)\n")

rows = []
for t in tasks[:30]:
    rows.append({
        "hash": t.path.name,
        "exitcode": t.exitcode,
        "failed": t.failed,
        "tag": (t.tag or "")[:80],
    })

try:
    import pandas as pd
    display(pd.DataFrame(rows))
except ImportError:
    for r in rows:
        print(r)

In [ ]:
failures = wu.failed_tasks(tasks)
if not failures:
    print("No directories with non-zero .exitcode found.")
    print("If the pipeline failed recently, check nextflow.log or pick a hash from the table above.")
else:
    print(f"{len(failures)} failed task(s):")
    for t in failures:
        print(f"  exit={t.exitcode}  {t.path}")

In [ ]:
# --- Pick a task hash directory to inspect ---
TASK_HASH = failures[0].path.name if failures else (tasks[0].path.name if tasks else None)
TASK_DIR = paths["work_dir"] / TASK_HASH if TASK_HASH else None

if TASK_DIR is None:
    raise RuntimeError("No task directories under work/. Run a pipeline first or fix RUN_DIR.")

print("Inspecting:", TASK_DIR)
artifacts = wu.command_artifacts(TASK_DIR)
for name, p in artifacts.items():
    print(f"  {name}: {p}")

In [ ]:
print("=== .command.err (tail) ===")
print(wu.tail_file(artifacts.get(".command.err"), n=100))

print("\n=== .command.sh (head) ===")
print(wu.peek_file(artifacts.get(".command.sh"), n=80))

In [ ]:
print("=== Published outputs (under run outputs/) ===")
tree = wu.list_outputs_tree(paths["outputs_dir"])
if not tree:
    print("(no outputs/ yet or directory missing)")
else:
    for line in tree[:80]:
        print(line)
    if len(tree) > 80:
        print(f"... and {len(tree) - 80} more paths")

In [ ]:
# Optional: peek staged .h5ad in the selected task dir (requires anndata in requirements)
staged = wu.find_staged_data_files(TASK_DIR)
print("Staged data files:", staged or "(none)")

h5ad_files = [p for p in staged if p.suffix == ".h5ad"]
if h5ad_files:
    try:
        import anndata as ad
        adata = ad.read_h5ad(h5ad_files[0])
        print(adata)
    except ImportError:
        print("Install anndata (see requirements-notebooks.txt) or use the R exploratory template for .rds.")
else:
    print("No .h5ad in this task dir. For Seurat .rds, use exploratory-analysis.Rmd with the R kernel.")

In [ ]:
# Optional: tail nextflow.log from the run
if paths["log_file"].is_file():
    print(wu.tail_file(paths["log_file"], n=60))
else:
    print("Log not found:", paths["log_file"])